# Phase 2A — seqproc vs splitcode downstream concordance (box dev)

Both tools run with **symmetric** configs (UMI 10 + bc3 8 + bc2 8 + bc1 8 = 34bp, observed barcodes), quantified with the **same** STARsolo `CB_UMI_Complex` config so all barcode error-correction is identical. Box-dev caveats: mapping is **chr19 only** on a **250k-read** subsample, so matrices are sparse; the cluster run uses full GRCm38 + full data + the split-pipe vendor.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../scripts'))
import numpy as np, matplotlib.pyplot as plt
from numpy import log1p, corrcoef
from concordance_helpers import load_star_raw, per_barcode_umi, barcode_rank, aligned_barcode_umi
RES = '../results/box_dev'

In [ ]:
sp_m, sp_bc, sp_g = load_star_raw(f'{RES}/seqproc_Solo.out/Gene')
sc_m, sc_bc, sc_g = load_star_raw(f'{RES}/splitcode_Solo.out/Gene')
print('seqproc  cells x genes:', sp_m.shape, '| nnz', sp_m.nnz)
print('splitcode cells x genes:', sc_m.shape, '| nnz', sc_m.nnz)

## Barcode-rank (knee) plot

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
for m, lbl, c in [(sp_m,'seqproc','#1565c0'), (sc_m,'splitcode','#e8743b')]:
    r = barcode_rank(m); r = r[r>0]
    ax.loglog(np.arange(1,len(r)+1), r, label=f'{lbl} (n={len(r)})', color=c, lw=1.5)
ax.set_xlabel('barcode rank'); ax.set_ylabel('total UMI per barcode')
ax.set_title('Barcode-rank (chr19, 250k reads)'); ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

## Summary statistics

In [ ]:
for m, bc, lbl in [(sp_m,sp_bc,'seqproc'), (sc_m,sc_bc,'splitcode')]:
    u = per_barcode_umi(m); nz = u[u>0]
    genes_det = int((np.asarray((m>0).sum(0)).ravel()>0).sum())
    print(f'{lbl:10s} barcodes_with_counts={len(nz):6d}  total_UMI={int(u.sum()):7d}  genes_detected={genes_det:5d}')

## Tool-vs-tool concordance: per-barcode UMI

In [ ]:
keys, a, b = aligned_barcode_umi(sp_bc, per_barcode_umi(sp_m), sc_bc, per_barcode_umi(sc_m))
mask = (a+b) > 0
r = corrcoef(log1p(a[mask]), log1p(b[mask]))[0,1] if mask.sum()>1 else float('nan')
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(log1p(a[mask]), log1p(b[mask]), s=10, alpha=0.5, color='#1565c0')
lim = max(log1p(a).max(), log1p(b).max(), 1); ax.plot([0,lim],[0,lim],'k--',lw=1)
ax.set_xlabel('seqproc  log1p(UMI/barcode)'); ax.set_ylabel('splitcode  log1p(UMI/barcode)')
ax.set_title(f'per-barcode UMI concordance   r = {r:.3f}')
plt.tight_layout(); plt.show()
print('shared barcodes (both>0):', int(((a>0)&(b>0)).sum()))

## Tool-vs-tool concordance: per-gene totals

In [ ]:
assert list(sp_g) == list(sc_g)
ga = np.asarray(sp_m.sum(0)).ravel(); gb = np.asarray(sc_m.sum(0)).ravel()
m = (ga+gb) > 0
r = corrcoef(log1p(ga[m]), log1p(gb[m]))[0,1] if m.sum()>1 else float('nan')
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(log1p(ga[m]), log1p(gb[m]), s=10, alpha=0.5, color='#2e7d32')
lim = max(log1p(ga).max(), log1p(gb).max(), 1); ax.plot([0,lim],[0,lim],'k--',lw=1)
ax.set_xlabel('seqproc  log1p(gene UMI)'); ax.set_ylabel('splitcode  log1p(gene UMI)')
ax.set_title(f'per-gene total concordance   r = {r:.3f}'); plt.tight_layout(); plt.show()

## Read-out

High per-barcode and per-gene correlation is the expected result, since the tools were made structurally symmetric and barcode correction is identical (STARsolo). Residual differences reflect only read-recovery differences. The cluster run repeats on full GRCm38 + full data and adds knee-based cell calling, ambient fraction, clustering ARI, cell typing, and the split-pipe vendor comparison.